# <span style="color: #a855f7">NumPy data analysis</span>

<span style="color: #a855f7">масиви, булева індексація, діагоналі, axis, concatenate, reshape з введенням даних через input

збережені виводи показують приклад запуску, Run All запитує значення заново, рандомні масиви генеряться при кожному запуску</span>

In [1]:
from html import escape

import numpy as np
from IPython.display import HTML, display

MAX_ELEMENTS = 100_000
INTEGER_LIMITS = np.iinfo(np.int64)
rng = np.random.default_rng()


def colorize(value):
    display(HTML(f'<pre style="color: #a855f7">{escape(str(value))}</pre>'))


def read_integer(prompt, minimum=INTEGER_LIMITS.min, maximum=INTEGER_LIMITS.max):
    while True:
        entered = input(prompt)
        try:
            value = int(entered)
        except ValueError:
            if not entered.strip():
                colorize("Empty input received; enter an integer")
            else:
                colorize(f"Invalid integer; received: {entered!a}")
            continue
        if minimum <= value <= maximum:
            return value
        colorize(f"Enter a value from {minimum} to {maximum}")


def read_range(required_size=None, require_negative=False):
    while True:
        lower = read_integer("Lower bound: ")
        upper = read_integer("Upper bound: ")
        if lower > upper:
            colorize("Lower bound must not exceed upper bound")
        elif require_negative and lower >= 0:
            colorize("Lower bound must be negative")
        elif required_size is not None and upper - lower + 1 != required_size:
            colorize(f"The inclusive range must contain exactly {required_size} integers")
        else:
            return lower, upper


def read_shape(expected_size=None):
    while True:
        rows = read_integer("Rows: ", minimum=1, maximum=MAX_ELEMENTS)
        columns = read_integer("Columns: ", minimum=1, maximum=MAX_ELEMENTS)
        size = rows * columns
        if size > MAX_ELEMENTS:
            colorize(f"The matrix must contain at most {MAX_ELEMENTS} elements")
        elif expected_size is not None and size != expected_size:
            colorize(f"The new shape must contain exactly {expected_size} elements")
        else:
            return rows, columns

## <span style="color: #a855f7">Threshold and sorting</span>

<span style="color: #a855f7">20 рандомних цілих чисел від 0 до 50 включно, к-сть значень > threshold, max, індекс першого max і сортування за спаданням, індекси починаються з 0</span>

In [2]:
values = rng.integers(0, 50, size=20, endpoint=True)
colorize(f"original: {values}")
threshold = read_integer("Threshold: ")
above_threshold = np.count_nonzero(values > threshold)
max_index = int(np.argmax(values))
descending = np.sort(values)[::-1]

colorize(f"threshold: {threshold}\nabove threshold: {above_threshold}")
colorize(f"max: {values.max()}\nfirst max index: {max_index}")
colorize(f"descending: {descending}")

## <span style="color: #a855f7">Diagonal and triangular matrix</span>

<span style="color: #a855f7">матриця 5×5 у введеному діапазоні включно з обома межами, головна діагональ, її sum і обнулення елементів строго вище діагоналі</span>

In [3]:
lower, upper = read_range()
diagonal_matrix = rng.integers(lower, upper, size=(5, 5), endpoint=True)
colorize(f"original:\n{diagonal_matrix}")
diagonal = np.diag(diagonal_matrix).copy()
diagonal_sum = diagonal.sum(dtype=object)

colorize(f"diagonal: {diagonal}")
colorize(f"diagonal sum: {diagonal_sum}")
diagonal_matrix[np.triu_indices(5, k=1)] = 0
colorize(f"modified:\n{diagonal_matrix}")

## <span style="color: #a855f7">Sequence and axis operations</span>

<span style="color: #a855f7">включний діапазон має містити рівно 30 цілих чисел для reshape у 6×5, суми рядків через axis=1 і max стовпців через axis=0</span>

In [4]:
sequence_rows, sequence_columns = 6, 5
sequence_size = sequence_rows * sequence_columns
start, end = read_range(required_size=sequence_size)
sequence = np.arange(sequence_size, dtype=np.int64) + start
sequence_matrix = sequence.reshape(sequence_rows, sequence_columns)
row_sums = sequence_matrix.sum(axis=1, dtype=object)
column_maxima = sequence_matrix.max(axis=0)

colorize(f"matrix:\n{sequence_matrix}")
colorize(f"row sums: {row_sums}")
colorize(f"column max: {column_maxima}")

## <span style="color: #a855f7">Negative values and array copy</span>

<span style="color: #a855f7">15 рандомних цілих чисел у включному діапазоні з від'ємною нижньою межею, булева маска для від'ємних значень, заміна їх нулями в копії і підрахунок нулів</span>

In [5]:
lower, upper = read_range(require_negative=True)
signed_values = rng.integers(lower, upper, size=15, endpoint=True)
negative_values = signed_values[signed_values < 0]
modified_values = signed_values.copy()
modified_values[modified_values < 0] = 0
zero_count = np.count_nonzero(modified_values == 0)

colorize(f"original: {signed_values}")
colorize(f"negative: {negative_values}")
colorize(f"modified: {modified_values}")
colorize(f"zeros: {zero_count}")

## <span style="color: #a855f7">Concatenation and elementwise operations</span>

<span style="color: #a855f7">два масиви введеної довжини, рандомні цілі значення від 0 до 10 і від 10 до 20 включно, concatenate, поелементні sum і difference, довжина від 1 до 100000</span>

In [6]:
length = read_integer("Array length: ", minimum=1, maximum=MAX_ELEMENTS)
first = rng.integers(0, 10, size=length, endpoint=True)
second = rng.integers(10, 20, size=length, endpoint=True)
combined = np.concatenate((first, second))
elementwise_sum = first + second
elementwise_difference = first - second

colorize(f"first: {first}")
colorize(f"second: {second}")
colorize(f"combined: {combined}")
colorize(f"sum: {elementwise_sum}")
colorize(f"difference: {elementwise_difference}")

## <span style="color: #a855f7">Reshape and row statistics</span>

<span style="color: #a855f7">початкова матриця з послідовністю від 0, нова форма з тою самою к-стю елементів, min і max рядків через axis=1, загальна sum, максимум 100000 елементів</span>

In [7]:
rows, columns = read_shape()
original_matrix = np.arange(rows * columns).reshape(rows, columns)
colorize(f"original:\n{original_matrix}")
colorize("New shape")
new_rows, new_columns = read_shape(expected_size=original_matrix.size)
reshaped_matrix = original_matrix.reshape(new_rows, new_columns)
row_minima = reshaped_matrix.min(axis=1)
row_maxima = reshaped_matrix.max(axis=1)
total = reshaped_matrix.sum()

colorize(f"reshaped:\n{reshaped_matrix}")
colorize(f"row min: {row_minima}")
colorize(f"row max: {row_maxima}")
colorize(f"total: {total}")